# The RunMultiple vs. ThreadPools Showdown
https://milescole.dev/data-engineering/2024/04/26/Fabric-Concurrency-Showdown-RunMultiple-vs-ThreadPool.html

In [1]:
import concurrent.futures
import pyspark.sql.functions as sf

spark.conf.set("spark.notebookutils.runmultiple.limit", 128)

instances = spark.conf.get("spark.executor.instances")
cores = spark.conf.get("spark.executor.cores")
print(f"{instances}x{cores}")

jobs = 10
concurrency = 2

StatementMeta(, b5fd5978-1638-4a5b-88d3-cf298e014d1d, 3, Finished, Available, Finished, False)

1x8


In [2]:
dag = {
    "activities": [
        {
            "name": f"Notebook{i}",
            "path": "RunMultiple_ChildJob",
            "timeoutPerCellInSeconds": 240,
            "args": {"job_id": i}
        }
        for i in range(1, jobs + 1)
    ],
    "concurrency": concurrency
}
mssparkutils.notebook.runMultiple(dag)

StatementMeta(, b5fd5978-1638-4a5b-88d3-cf298e014d1d, 4, Submitted, Running, Running, True)

In [ ]:
def write_job(job_id):
    df = spark.range(100)   # 1000000
    df = df.withColumn("sk", sf.monotonically_increasing_id()).withColumn("instance_id", sf.lit(job_id)).drop("id")
    df.write.mode("overwrite").parquet(f"abfss://<your workspace>@onelake.dfs.fabric.microsoft.com/<your lakehouse>/Files/benchmark/multithreading/{job_id}/")
    print(f"done: {job_id}")

with concurrent.futures.ThreadPoolExecutor(max_workers=concurrency) as executor:
    # Submit all jobs to the executor
    futures = [executor.submit(write_job, job_id) for job_id in range(jobs)]


StatementMeta(, , -1, Waiting, , Waiting, True)